# 02. Modelo Predictivo Interno

**Solventa - Prueba Técnica Data Scientist Jr.**

Entrenamiento, evaluación de modelos y selección del punto de corte.

5. ENTRENAMIENTO DE MODELOS

In [ ]:
print("\n[5] Entrenando modelos...")

models = {
    "Gradient Boosting": GradientBoostingClassifier(
        n_estimators=200,
        max_depth=4,
        learning_rate=0.05,
        min_samples_split=20,
        min_samples_leaf=10,
        subsample=0.8,
        random_state=42,
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        max_depth=8,
        min_samples_split=20,
        min_samples_leaf=10,
        random_state=42,
        class_weight="balanced",
    ),
    "Logistic Regression": LogisticRegression(
        max_iter=1000, C=0.1, class_weight="balanced", random_state=42
    ),
}

results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    y_prob = model.predict_proba(X_test)[:, 1]
    auc = roc_auc_score(y_test, y_prob)
    results[name] = {"model": model, "y_prob": y_prob, "auc": auc}
    print(f"   {name}: ROC-AUC = {auc:.4f}")

best_model_name = max(results, key=lambda k: results[k]["auc"])
best_model = results[best_model_name]["model"]
best_probs = results[best_model_name]["y_prob"]
print(
    f"\n   Mejor modelo: {best_model_name} (ROC-AUC: {results[best_model_name]['auc']:.4f})"
)

6. CURVAS ROC Y PR

In [ ]:
print("\n[6] Generando curvas ROC y Precision-Recall...")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ROC Curve
for name, res in results.items():
    fpr, tpr, _ = roc_curve(y_test, res["y_prob"])
    axes[0].plot(fpr, tpr, label=f"{name} (AUC={res['auc']:.4f})", linewidth=2)
axes[0].plot([0, 1], [0, 1], "k--", linewidth=1)
axes[0].set_xlabel("Tasa de Falsos Positivos")
axes[0].set_ylabel("Tasa de Verdaderos Positivos")
axes[0].set_title("Curva ROC - Comparación de Modelos")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Precision-Recall Curve
for name, res in results.items():
    precision, recall, _ = precision_recall_curve(y_test, res["y_prob"])
    axes[1].plot(recall, precision, label=name, linewidth=2)
axes[1].set_xlabel("Recall")
axes[1].set_ylabel("Precision")
axes[1].set_title("Curva Precision-Recall")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("output/figures/04_roc_pr_curves.png", dpi=150, bbox_inches="tight")
plt.close()
print("   -> Guardado: 04_roc_pr_curves.png")

7. IMPORTANCIA DE VARIABLES

In [ ]:
print("\n[7] Analizando importancia de variables...")

# Feature importance del mejor modelo
if hasattr(best_model, "feature_importances_"):
    importances = best_model.feature_importances_
elif hasattr(best_model, "coef_"):
    importances = np.abs(best_model.coef_[0])
else:
    importances = np.zeros(len(feature_cols))

feat_imp = pd.DataFrame(
    {"Feature": feature_cols, "Importance": importances}
).sort_values("Importance", ascending=True)

fig, ax = plt.subplots(figsize=(10, 8))
ax.barh(feat_imp["Feature"], feat_imp["Importance"], color="#3498db")
ax.set_xlabel("Importancia")
ax.set_title(f"Importancia de Variables - {best_model_name}")
plt.tight_layout()
plt.savefig("output/figures/05_feature_importance.png", dpi=150, bbox_inches="tight")
plt.close()
print("   -> Guardado: 05_feature_importance.png")

8. SELECCIÓN DE PUNTO DE CORTE

In [ ]:
print("\n[8] Selección de punto de corte...")

thresholds = np.arange(0.1, 0.9, 0.01)
metrics = []

for t in thresholds:
    y_pred_t = (best_probs >= t).astype(int)
    metrics.append(
        {
            "threshold": t,
            "accuracy": accuracy_score(y_test, y_pred_t),
            "precision": precision_score(y_test, y_pred_t, zero_division=0),
            "recall": recall_score(y_test, y_pred_t, zero_division=0),
            "f1": f1_score(y_test, y_pred_t, zero_division=0),
            "predicted_default": y_pred_t.sum(),
            "predicted_approved": (y_pred_t == 0).sum(),
            "approval_rate": (y_pred_t == 0).sum() / len(y_test),
        }
    )

metrics_df = pd.DataFrame(metrics)

# Encontrar punto de corte óptimo (max F1)
optimal_idx = metrics_df["f1"].idxmax()
optimal_threshold = metrics_df.loc[optimal_idx, "threshold"]

# Para un producto de ALTO RIESGO, buscamos un threshold que balancee
# captura de malos sin rechazar demasiados buenos.
# Criterio: maximizar Youden's J (sensibilidad + especificidad - 1)

In [ ]:
from sklearn.metrics import roc_curve as sk_roc_curve

In [ ]:
fpr_t, tpr_t, thresholds_t = sk_roc_curve(y_test, best_probs)
youden_j = tpr_t - fpr_t
youden_threshold = thresholds_t[np.argmax(youden_j)]

# Buscar threshold que capture al menos 50% de malos aprobando al menos 50% de buenos
recommended_threshold = youden_threshold
for t in sorted(thresholds):
    y_pred_t = (best_probs >= t).astype(int)
    recall_t = recall_score(y_test, y_pred_t, zero_division=0)
    approval_t = (y_pred_t == 0).sum() / len(y_test)
    if recall_t >= 0.50 and approval_t >= 0.50:
        recommended_threshold = t
        break

y_pred_final = (best_probs >= recommended_threshold).astype(int)

print(f"   Threshold F1-max: {optimal_threshold:.2f}")
print(f"   Threshold Youden's J: {youden_threshold:.2f}")
print(f"   Threshold recomendado (alto riesgo): {recommended_threshold:.2f}")
print(
    f"   Tasa de aprobación: {metrics_df[metrics_df['threshold'] == recommended_threshold]['approval_rate'].values[0] * 100:.1f}%"
)
print(
    f"   Precision a {recommended_threshold:.2f}: {precision_score(y_test, y_pred_final):.4f}"
)
print(
    f"   Recall a {recommended_threshold:.2f}: {recall_score(y_test, y_pred_final):.4f}"
)
print(f"   Threshold recomendado (alto riesgo): {recommended_threshold:.2f}")
print(
    f"   Tasa de aprobación: {metrics_df[metrics_df['threshold'] == recommended_threshold]['approval_rate'].values[0] * 100:.1f}%"
)
print(
    f"   Precision a {recommended_threshold:.2f}: {precision_score(y_test, y_pred_final):.4f}"
)
print(
    f"   Recall a {recommended_threshold:.2f}: {recall_score(y_test, y_pred_final):.4f}"
)

9. ANÁLISIS DEL PUNTO DE CORTE

In [ ]:
print("\n[9] Visualizando análisis de punto de corte...")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Metrics vs threshold
axes[0].plot(
    metrics_df["threshold"], metrics_df["precision"], label="Precision", linewidth=2
)
axes[0].plot(metrics_df["threshold"], metrics_df["recall"], label="Recall", linewidth=2)
axes[0].plot(metrics_df["threshold"], metrics_df["f1"], label="F1-Score", linewidth=2)
axes[0].axvline(
    x=recommended_threshold,
    color="red",
    linestyle="--",
    label=f"Corte={recommended_threshold:.2f}",
)
axes[0].axvline(
    x=optimal_threshold,
    color="green",
    linestyle=":",
    label=f"F1-max={optimal_threshold:.2f}",
)
axes[0].set_xlabel("Threshold")
axes[0].set_ylabel("Valor")
axes[0].set_title("Métricas vs Punto de Corte")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Approval rate vs threshold
ax2 = axes[1]
ax2.plot(
    metrics_df["threshold"],
    metrics_df["approval_rate"] * 100,
    color="#2ecc71",
    linewidth=2,
)
ax2.axvline(
    x=recommended_threshold,
    color="red",
    linestyle="--",
    label=f"Corte={recommended_threshold:.2f}",
)
ax2.set_xlabel("Threshold")
ax2.set_ylabel("Tasa de Aprobación (%)", color="#2ecc71")
ax2.set_title("Tasa de Aprobación vs Punto de Corte")
ax2.legend(loc="upper right")
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("output/figures/06_cutoff_analysis.png", dpi=150, bbox_inches="tight")
plt.close()
print("   -> Guardado: 06_cutoff_analysis.png")

10. MATRIZ DE CONFUSIÓN

In [ ]:
print("\n[10] Matriz de confusión...")

cm = confusion_matrix(y_test, y_pred_final)
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    ax=ax,
    xticklabels=["Aprobado", "Rechazado"],
    yticklabels=["No Mora", "Mora 30+"],
)
ax.set_xlabel("Predicción")
ax.set_ylabel("Real")
ax.set_title(
    f"Matriz de Confusión - {best_model_name} (corte={recommended_threshold:.2f})"
)
plt.tight_layout()
plt.savefig("output/figures/07_confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.close()
print("   -> Guardado: 07_confusion_matrix.png")

11. CALIBRACIÓN

In [ ]:
print("\n[11] Curva de calibración...")

fig, ax = plt.subplots(figsize=(6, 5))
frac_of_pos, mean_pred_val = calibration_curve(
    y_test, best_probs, n_bins=10, strategy="uniform"
)
ax.plot(mean_pred_val, frac_of_pos, "s-", label=best_model_name, markersize=8)
ax.plot([0, 1], [0, 1], "k:", label="Perfecto")
ax.set_xlabel("Probabilidad predicha")
ax.set_ylabel("Fracción de positivos")
ax.set_title("Curva de Calibración")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("output/figures/08_calibration.png", dpi=150, bbox_inches="tight")
plt.close()
print("   -> Guardado: 08_calibration.png")

12. REPORTE FINAL

In [ ]:
print("\n" + "=" * 60)
print("RESUMEN DEL MODELO")
print("=" * 60)

print(f"\nModelo seleccionado: {best_model_name}")
print(f"ROC-AUC: {results[best_model_name]['auc']:.4f}")
print(f"Punto de corte: {recommended_threshold:.2f}")
print(f"\nMétricas en test:")
print(f"  Accuracy:  {accuracy_score(y_test, y_pred_final):.4f}")
print(f"  Precision: {precision_score(y_test, y_pred_final):.4f}")
print(f"  Recall:    {recall_score(y_test, y_pred_final):.4f}")
print(f"  F1-Score:  {f1_score(y_test, y_pred_final):.4f}")
print(f"\nTasa de aprobación: {(y_pred_final == 0).sum() / len(y_test) * 100:.1f}%")
print(f"Clientes rechazados: {y_pred_final.sum()} de {len(y_test)}")

print(
    "\n"
    + classification_report(y_test, y_pred_final, target_names=["No Mora", "Mora 30+"])
)

# Guardar modelo
with open("output/model.pkl", "wb") as f:
    pickle.dump(
        {
            "model": best_model,
            "feature_cols": feature_cols,
            "label_encoders": label_encoders,
            "threshold": recommended_threshold,
            "model_name": best_model_name,
        },
        f,
    )
print("Modelo guardado en output/model.pkl")

# Guardar métricas para reporte
report_metrics = {
    "best_model": best_model_name,
    "roc_auc": results[best_model_name]["auc"],
    "threshold": recommended_threshold,
    "accuracy": accuracy_score(y_test, y_pred_final),
    "precision": precision_score(y_test, y_pred_final),
    "recall": recall_score(y_test, y_pred_final),
    "f1": f1_score(y_test, y_pred_final),
    "approval_rate": (y_pred_final == 0).sum() / len(y_test),
    "mora30_rate": y.mean(),
    "mora60_rate": df["Mora60"].mean(),
    "feature_importance": feat_imp.to_dict("records"),
    "all_model_auc": {name: res["auc"] for name, res in results.items()},
    "confusion_matrix": cm.tolist(),
    "metrics_by_threshold": metrics_df.to_dict("records"),
}

In [ ]:
import json

In [ ]:
with open("output/model_metrics.json", "w") as f:
    json.dump(report_metrics, f, indent=2)

print("\nParte 1 completada exitosamente!")